In [1]:
import json
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import re
import os

# ============================================
# КОНФИГУРАЦИЯ
# ============================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   
OUTPUT_DIR = "./personality_model_qwen"
TRAIN_FILE = "train_prompts_0shot_clean.jsonl"
VAL_FILE = "val_prompts_0shot_clean.jsonl"
TEST_FILE = "test_prompts_0shot_clean.jsonl"

BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 5
MAX_LENGTH = 1536                
LOGGING_STEPS = 10
SAVE_STEPS = 100
EVAL_STEPS = 100

# ============================================
# ШАГ 1: Загрузка данных
# ============================================
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

train_data = load_jsonl(TRAIN_FILE)
val_data = load_jsonl(VAL_FILE)
test_data = load_jsonl(TEST_FILE)
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# ============================================
# ШАГ 2: Подготовка примеров 
# ============================================
def create_training_example(item):
    prompt = item['prompt']
    true_labels = item['true_labels_russian']   # это строки 'низкий','средний','высокий'
    completion = json.dumps(true_labels, ensure_ascii=False)
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": completion}
    ]
    return {
        "messages": messages,
        "true_labels": true_labels
    }

train_examples = [create_training_example(item) for item in train_data]
val_examples = [create_training_example(item) for item in val_data]
test_examples = [create_training_example(item) for item in test_data]

# ============================================
# ШАГ 3: Токенизация с использованием chat_template
# ============================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenized = tokenizer.apply_chat_template(
        examples["messages"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_dict=True
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)
test_dataset = Dataset.from_list(test_examples)

tokenized_train = train_dataset.map(tokenize_function, batched=True,
                                    remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(tokenize_function, batched=True,
                                remove_columns=val_dataset.column_names)
tokenized_test = test_dataset.map(tokenize_function, batched=True,
                                  remove_columns=test_dataset.column_names)

dataset_dict = DatasetDict({
    'train': tokenized_train,
    'validation': tokenized_val,
    'test': tokenized_test
})

print("Пример начала токенизированного промпта:")
print(tokenizer.decode(tokenized_train[0]['input_ids'], skip_special_tokens=False)[:400])

# ============================================
# ШАГ 4: Модель + LoRA
# ============================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================
# ШАГ 5: Коллбэк для метрик 
# ============================================
class PersonalityMetricsCallback:
    def __init__(self, tokenizer, val_examples):
        self.tokenizer = tokenizer
        self.val_examples = val_examples

    def extract_json(self, text):
        m = re.search(r'\{\s*"открытость_опыту".*?\}', text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
        for m in re.finditer(r'\{.*?\}', text, re.DOTALL):
            try:
                obj = json.loads(m.group())
                if 'открытость_опыту' in obj:
                    return obj
            except:
                continue
        return None

    def compute_metrics(self, model):
        model.eval()
        predictions, targets = [], []
        for i in range(min(20, len(self.val_examples))):
            example = self.val_examples[i]
            prompt = self.tokenizer.apply_chat_template(
                example["messages"][:1], 
                add_generation_prompt=True,
                tokenize=False
            )
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True,
                                    max_length=MAX_LENGTH).to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            pred = self.extract_json(generated)
            if pred:
                predictions.append(pred)
                targets.append(example["true_labels"])

        model.train()
        if not predictions:
            return {"accuracy": 0.0, "f1": 0.0}

        traits = ['открытость_опыту', 'добросовестность', 'экстраверсия',
                  'доброжелательность', 'нейротизм']
        accs, f1s = [], []
        for t in traits:
            yt = [tr[t] for tr in targets]
            yp = [p.get(t, -1) for p in predictions]
            valid = [i for i, v in enumerate(yp) if v != -1]
            if valid:
                ytv = [yt[i] for i in valid]
                ypv = [yp[i] for i in valid]
                accs.append(accuracy_score(ytv, ypv))
                f1s.append(f1_score(ytv, ypv, average='weighted'))
        return {
            "accuracy": np.mean(accs) if accs else 0.0,
            "f1": np.mean(f1s) if f1s else 0.0,
            "predictions_made": len(predictions)
        }

metrics_callback = PersonalityMetricsCallback(tokenizer, val_examples)

# ============================================
# ШАГ 6: Trainer
# ============================================
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,
    save_total_limit=3,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict['train'],
    eval_dataset=dataset_dict['validation'],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# ============================================
# ШАГ 7: Обучение
# ============================================
print("Начало обучения...")
trainer.train()
model.save_pretrained(f"{OUTPUT_DIR}/lora_weights")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_weights")

# ============================================
# ШАГ 8: Тестирование
# ============================================
def test_model(model, test_examples, tokenizer, num_samples=30):
    model.eval()
    results = []
    for i in range(min(num_samples, len(test_examples))):
        example = test_examples[i]
        prompt = tokenizer.apply_chat_template(
            example["messages"][:1],
            add_generation_prompt=True,
            tokenize=False
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=MAX_LENGTH).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = PersonalityMetricsCallback.extract_json(None, generated)
        results.append({
            "user_id": test_data[i].get("user_id", i),
            "true": example["true_labels"],
            "predicted": pred,
            "generated_text": generated
        })
        if i < 3:
            print(f"\n--- Пример {i+1} ---")
            print("Сгенерировано:", generated[-200:])
            print("Извлечённый JSON:", pred)
    return results

print("\nТестирование...")
test_results = test_model(model, test_examples, tokenizer, num_samples=30)

correct = total = 0
traits = ['открытость_опыту', 'добросовестность', 'экстраверсия', 'доброжелательность', 'нейротизм']
trait_correct = {t: 0 for t in traits}
for res in test_results:
    if res["predicted"]:
        for t in traits:
            total += 1
            if res["true"].get(t) == res["predicted"].get(t):
                correct += 1
                trait_correct[t] += 1

print(f"\nОбщая точность: {correct/total:.2%}" if total else "Нет предсказаний")
for t in traits:
    print(f"  {t}: {trait_correct[t]}/{len(test_results)}")

with open(f"{OUTPUT_DIR}/test_results.json", 'w', encoding='utf-8') as f:
    json.dump(test_results, f, ensure_ascii=False, indent=2)

Train: 619, Val: 133, Test: 133


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Начало обучения...


Step,Training Loss,Validation Loss
100,0.721369,0.759520
200,0.680769,0.758357
300,0.573331,0.785225


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Тестирование...

--- Пример 1 ---
Сгенерировано:  "добросовестность": 1, "экстраверсия": 2, "доброжелательность": 0, "нейротизм": 1}
assistant
{"открытость_опыту": 0, "добросовестность": 1, "экстраверсия": 1, "доброжелательность": 1, "нейротизм": 2}
Извлечённый JSON: {'открытость_опыту': 0, 'добросовестность': 1, 'экстраверсия': 2, 'доброжелательность': 0, 'нейротизм': 1}

--- Пример 2 ---
Сгенерировано:  "добросовестность": 1, "экстраверсия": 2, "доброжелательность": 0, "нейротизм": 1}
assistant
{"открытость_опыту": 0, "добросовестность": 0, "экстраверсия": 0, "доброжелательность": 0, "нейротизм": 2}
Извлечённый JSON: {'открытость_опыту': 0, 'добросовестность': 1, 'экстраверсия': 2, 'доброжелательность': 0, 'нейротизм': 1}

--- Пример 3 ---
Сгенерировано:  "добросовестность": 1, "экстраверсия": 2, "доброжелательность": 0, "нейротизм": 1}
assistant
{"открытость_опыту": 0, "добросовестность": 0, "экстраверсия": 0, "доброжелательность": 0, "нейротизм": 2}
Извлечённый JSON: {'открытость_

In [1]:
import json
import re
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)
import pandas as pd
from tqdm import tqdm

# ============================================
# КОНФИГУРАЦИЯ
# ============================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_WEIGHTS_PATH = "./personality_model_qwen/lora_weights"  # путь к сохранённым весам
TEST_FILE = "test_prompts_0shot_clean.jsonl"
MAX_LENGTH = 2048  # должно совпадать с обучением
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================
# ШАГ 1: Загрузка данных
# ============================================
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

test_data = load_jsonl(TEST_FILE)
print(f"Загружено тестовых примеров: {len(test_data)}")

# ============================================
# ШАГ 2: Загрузка модели и токенизатора
# ============================================
print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Загрузка LoRA весов...")
model = PeftModel.from_pretrained(base_model, LORA_WEIGHTS_PATH)
model.eval()
print("Модель загружена успешно!")

# ============================================
# ШАГ 3: Функции для генерации и извлечения ответов
# ============================================
def extract_json(text):
    """Извлечение JSON из сгенерированного текста"""
    # Сначала ищем полный JSON с ключами модели
    m = re.search(r'\{\s*"открытость_опыту".*?\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass
    
    # Ищем любой JSON объект
    for m in re.finditer(r'\{.*?\}', text, re.DOTALL):
        try:
            obj = json.loads(m.group())
            if 'открытость_опыту' in obj:
                return obj
        except:
            continue
    return None

def predict_single(model, tokenizer, prompt, max_new_tokens=100):
    """Генерация ответа для одного примера"""
    # Форматируем промпт с chat_template
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    
    inputs = tokenizer(
        formatted_prompt, 
        return_tensors="pt", 
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Декодируем только новые токены
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return generated_text, extract_json(generated_text)

# ============================================
# ШАГ 4: Предсказание на всём тестовом наборе
# ============================================
print("\nГенерация предсказаний для тестового набора...")
results = []
failed_predictions = 0

for item in tqdm(test_data):
    prompt = item['prompt']
    true_labels = item['true_labels_russian']
    
    generated_text, predicted_json = predict_single(model, tokenizer, prompt)
    
    if predicted_json is None:
        failed_predictions += 1
        # Создаём пустой словарь для неудачных предсказаний
        predicted_json = {trait: None for trait in true_labels.keys()}
    
    results.append({
        "user_id": item.get("user_id", ""),
        "true_labels": true_labels,
        "predicted_labels": predicted_json,
        "generated_text": generated_text
    })

print(f"Успешных предсказаний: {len(results) - failed_predictions}/{len(results)}")
print(f"Неудачных предсказаний: {failed_predictions}")

# ============================================
# ШАГ 5: Подготовка данных для метрик
# ============================================
TRAITS = ['открытость_опыту', 'добросовестность', 'экстраверсия', 
          'доброжелательность', 'нейротизм']
LABEL_MAP = {'низкий': 0, 'средний': 1, 'высокий': 2}

# Собираем истинные и предсказанные метки для каждой черты
true_labels_dict = {trait: [] for trait in TRAITS}
pred_labels_dict = {trait: [] for trait in TRAITS}

for result in results:
    for trait in TRAITS:
        true_val = result['true_labels'].get(trait)
        pred_val = result['predicted_labels'].get(trait)
        
        # Конвертируем строки в числа
        true_num = LABEL_MAP.get(true_val)
        pred_num = LABEL_MAP.get(pred_val)
        
        true_labels_dict[trait].append(true_num)
        pred_labels_dict[trait].append(pred_num)

# ============================================
# ШАГ 6: Расчёт метрик
# ============================================
print("\n" + "="*80)
print("МЕТРИКИ ОЦЕНКИ КЛАССИФИКАЦИИ")
print("="*80)

# 6.1 Общая точность по всем чертам
print("\n--- Общие метрики ---")
all_true = []
all_pred = []

for trait in TRAITS:
    for true_val, pred_val in zip(true_labels_dict[trait], pred_labels_dict[trait]):
        if true_val is not None and pred_val is not None:
            all_true.append(true_val)
            all_pred.append(pred_val)

overall_accuracy = accuracy_score(all_true, all_pred)
print(f"Общая точность (Accuracy): {overall_accuracy:.4f}")

# 6.2 Коэффициент Каппа Коэна
kappa = cohen_kappa_score(all_true, all_pred)
print(f"Коэффициент Каппа Коэна: {kappa:.4f}")

# 6.3 Weighted Macro-averaged F1
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    all_true, all_pred, average='macro'
)
print(f"Macro F1: {f1_macro:.4f}")
print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")

# 6.4 Weighted F1
_, _, f1_weighted, _ = precision_recall_fscore_support(
    all_true, all_pred, average='weighted'
)
print(f"Weighted F1: {f1_weighted:.4f}")

# ============================================
# ШАГ 7: Метрики по каждой черте отдельно
# ============================================
print("\n" + "="*80)
print("МЕТРИКИ ПО КАЖДОЙ ЧЕРТЕ ЛИЧНОСТИ")
print("="*80)

trait_metrics = {}

for trait in TRAITS:
    true_vals = [v for v in true_labels_dict[trait] if v is not None]
    pred_vals = [v for v in pred_labels_dict[trait] if v is not None]
    
    # Фильтруем пары, где оба значения не None
    valid_pairs = [(t, p) for t, p in zip(true_labels_dict[trait], pred_labels_dict[trait])
                   if t is not None and p is not None]
    
    if not valid_pairs:
        print(f"\n{trait}: недостаточно данных")
        continue
    
    true_filtered = [t for t, p in valid_pairs]
    pred_filtered = [p for t, p in valid_pairs]
    
    acc = accuracy_score(true_filtered, pred_filtered)
    
    print(f"\n--- {trait} ---")
    print(f"Точность (Accuracy): {acc:.4f}")
    
    # Precision, Recall, F1 для каждого класса
    print("\nClassification Report:")
    print(classification_report(
        true_filtered, 
        pred_filtered, 
        target_names=['низкий (0)', 'средний (1)', 'высокий (2)'],
        zero_division=0
    ))
    
    # Матрица ошибок
    cm = confusion_matrix(true_filtered, pred_filtered, labels=[0, 1, 2])
    print("Матрица ошибок:")
    print(pd.DataFrame(cm, 
                       index=['True: низкий', 'True: средний', 'True: высокий'],
                       columns=['Pred: низкий', 'Pred: средний', 'Pred: высокий']))
    
    trait_metrics[trait] = {
        'accuracy': acc,
        'confusion_matrix': cm.tolist()
    }

# ============================================
# ШАГ 8: Сводная таблица результатов
# ============================================
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА")
print("="*80)

summary_data = []
for trait in TRAITS:
    true_vals = [v for v in true_labels_dict[trait] if v is not None]
    pred_vals = [v for v in pred_labels_dict[trait] if v is not None]
    
    valid_pairs = [(t, p) for t, p in zip(true_labels_dict[trait], pred_labels_dict[trait])
                   if t is not None and p is not None]
    
    if valid_pairs:
        true_filtered = [t for t, p in valid_pairs]
        pred_filtered = [p for t, p in valid_pairs]
        
        acc = accuracy_score(true_filtered, pred_filtered)
        prec, rec, f1, _ = precision_recall_fscore_support(
            true_filtered, pred_filtered, average='macro', zero_division=0
        )
    else:
        acc = prec = rec = f1 = 0.0
    
    summary_data.append({
        'Черта': trait,
        'Accuracy': acc,
        'Precision (macro)': prec,
        'Recall (macro)': rec,
        'F1 (macro)': f1
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# ============================================
# ШАГ 9: Сохранение результатов
# ============================================
output_results = {
    'overall_metrics': {
        'accuracy': overall_accuracy,
        'cohen_kappa': kappa,
        'macro_f1': f1_macro,
        'weighted_f1': f1_weighted,
        'macro_precision': precision_macro,
        'macro_recall': recall_macro
    },
    'trait_metrics': trait_metrics,
    'predictions': []
}

# Добавляем предсказания
for i, result in enumerate(results):
    pred_clean = {}
    for trait in TRAITS:
        pred_clean[trait] = result['predicted_labels'].get(trait)
    
    output_results['predictions'].append({
        'user_id': result['user_id'],
        'true_labels': result['true_labels'],
        'predicted_labels': pred_clean,
        'correct': {
            trait: result['true_labels'].get(trait) == result['predicted_labels'].get(trait)
            for trait in TRAITS
        }
    })

# Сохраняем результаты
output_path = f"{LORA_WEIGHTS_PATH}/../test_metrics.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output_results, f, ensure_ascii=False, indent=2)

print(f"\nРезультаты сохранены в {output_path}")

# ============================================
# ШАГ 10: Примеры предсказаний
# ============================================
print("\n" + "="*80)
print("ПРИМЕРЫ ПРЕДСКАЗАНИЙ (первые 5)")
print("="*80)

for i in range(min(5, len(results))):
    print(f"\n--- Пример {i+1} ---")
    print(f"User ID: {results[i]['user_id']}")
    print("Истинные метки:")
    for trait in TRAITS:
        print(f"  {trait}: {results[i]['true_labels'].get(trait)}")
    print("Предсказанные метки:")
    for trait in TRAITS:
        true_val = results[i]['true_labels'].get(trait)
        pred_val = results[i]['predicted_labels'].get(trait)
        match = "✓" if true_val == pred_val else "✗"
        print(f"  {trait}: {pred_val} {match}")
    print(f"Сгенерированный текст: {results[i]['generated_text'][:200]}...")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).
W0510 13:46:14.382000 26064 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Загружено тестовых примеров: 133
Загрузка токенизатора...
Загрузка базовой модели...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Загрузка LoRA весов...
Модель загружена успешно!

Генерация предсказаний для тестового набора...


100%|████████████████████████████████████████████████████████████████████████████████| 133/133 [14:59<00:00,  6.76s/it]

Успешных предсказаний: 133/133
Неудачных предсказаний: 0

МЕТРИКИ ОЦЕНКИ КЛАССИФИКАЦИИ

--- Общие метрики ---
Общая точность (Accuracy): nan
Коэффициент Каппа Коэна: nan
Macro F1: nan
Macro Precision: nan
Macro Recall: nan
Weighted F1: nan

МЕТРИКИ ПО КАЖДОЙ ЧЕРТЕ ЛИЧНОСТИ

открытость_опыту: недостаточно данных

добросовестность: недостаточно данных

экстраверсия: недостаточно данных

доброжелательность: недостаточно данных

нейротизм: недостаточно данных

СВОДНАЯ ТАБЛИЦА
             Черта  Accuracy  Precision (macro)  Recall (macro)  F1 (macro)
  открытость_опыту       0.0                0.0             0.0         0.0
  добросовестность       0.0                0.0             0.0         0.0
      экстраверсия       0.0                0.0             0.0         0.0
доброжелательность       0.0                0.0             0.0         0.0
         нейротизм       0.0                0.0             0.0         0.0

Результаты сохранены в ./personality_model_qwen/lora_weights/../te


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\numpy\lib\function_base.py:520: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Soft\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [2]:
import json
import re
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)
import pandas as pd
from tqdm import tqdm

# ============================================
# КОНФИГУРАЦИЯ
# ============================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_WEIGHTS_PATH = "./personality_model_qwen/lora_weights"
TEST_FILE = "test_prompts_0shot_clean.jsonl"
MAX_LENGTH = 2048
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================
# ШАГ 1: Загрузка данных
# ============================================
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

test_data = load_jsonl(TEST_FILE)
print(f"Загружено тестовых примеров: {len(test_data)}")

# Проверим формат данных
print("\nПример структуры данных:")
print(json.dumps(test_data[0], ensure_ascii=False, indent=2))

# ============================================
# ШАГ 2: Загрузка модели и токенизатора
# ============================================
print("\nЗагрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Загрузка LoRA весов...")
model = PeftModel.from_pretrained(base_model, LORA_WEIGHTS_PATH)
model.eval()
print("Модель загружена успешно!")

# ============================================
# ШАГ 3: Функции для генерации и извлечения ответов
# ============================================
def extract_json(text):
    """Извлечение JSON из сгенерированного текста"""
    # Сначала ищем полный JSON с ключами модели
    m = re.search(r'\{\s*"открытость_опыту".*?\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass
    
    # Ищем любой JSON объект
    for m in re.finditer(r'\{.*?\}', text, re.DOTALL):
        try:
            obj = json.loads(m.group())
            if 'открытость_опыту' in obj:
                return obj
        except:
            continue
    return None

def predict_single(model, tokenizer, prompt, max_new_tokens=100):
    """Генерация ответа для одного примера"""
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    
    inputs = tokenizer(
        formatted_prompt, 
        return_tensors="pt", 
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return generated_text, extract_json(generated_text)

# ============================================
# ШАГ 4: Предсказание на всём тестовом наборе
# ============================================
print("\nГенерация предсказаний для тестового набора...")
results = []
failed_predictions = 0

for item in tqdm(test_data):
    prompt = item['prompt']
    true_labels = item['true_labels_russian']
    
    generated_text, predicted_json = predict_single(model, tokenizer, prompt)
    
    if predicted_json is None:
        failed_predictions += 1
        predicted_json = {trait: None for trait in true_labels.keys()}
    
    results.append({
        "user_id": item.get("user_id", ""),
        "true_labels": true_labels,
        "predicted_labels": predicted_json,
        "generated_text": generated_text
    })

print(f"Успешных предсказаний: {len(results) - failed_predictions}/{len(results)}")
print(f"Неудачных предсказаний: {failed_predictions}")

# ============================================
# ШАГ 5: Подготовка данных для метрик
# ============================================
TRAITS = ['открытость_опыту', 'добросовестность', 'экстраверсия', 
          'доброжелательность', 'нейротизм']

# Анализируем формат истинных меток
print("\nАнализ формата истинных меток:")
first_true = results[0]['true_labels']
print(f"Тип значений: {type(list(first_true.values())[0])}")
print(f"Пример значений: {list(first_true.values())[:3]}")

# Создаем маппинг для конвертации
def normalize_label(label):
    """Конвертирует метку в число 0, 1, 2 независимо от формата"""
    if isinstance(label, (int, float)):
        return int(label)
    if isinstance(label, str):
        label_lower = label.lower().strip()
        if label_lower in ['низкий', 'low', '0']:
            return 0
        elif label_lower in ['средний', 'medium', '1']:
            return 1
        elif label_lower in ['высокий', 'high', '2']:
            return 2
    return None

# Собираем истинные и предсказанные метки
true_labels_dict = {trait: [] for trait in TRAITS}
pred_labels_dict = {trait: [] for trait in TRAITS}

for result in results:
    for trait in TRAITS:
        true_val = normalize_label(result['true_labels'].get(trait))
        pred_val = normalize_label(result['predicted_labels'].get(trait))
        
        true_labels_dict[trait].append(true_val)
        pred_labels_dict[trait].append(pred_val)

# Проверяем, что данные корректны
print("\nПроверка конвертированных данных:")
for trait in TRAITS[:1]:  # Проверяем первую черту
    valid_true = [v for v in true_labels_dict[trait] if v is not None]
    valid_pred = [v for v in pred_labels_dict[trait] if v is not None]
    print(f"{trait}:")
    print(f"  Валидных истинных: {len(valid_true)}")
    print(f"  Валидных предсказанных: {len(valid_pred)}")
    if valid_true:
        print(f"  Уникальные истинные: {set(valid_true)}")
    if valid_pred:
        print(f"  Уникальные предсказанные: {set(valid_pred)}")

# ============================================
# ШАГ 6: Расчёт метрик
# ============================================
print("\n" + "="*80)
print("МЕТРИКИ ОЦЕНКИ КЛАССИФИКАЦИИ")
print("="*80)

# Фильтруем только валидные пары
all_true = []
all_pred = []

for trait in TRAITS:
    for true_val, pred_val in zip(true_labels_dict[trait], pred_labels_dict[trait]):
        if true_val is not None and pred_val is not None:
            all_true.append(true_val)
            all_pred.append(pred_val)

print(f"Всего валидных пар: {len(all_true)}")

if len(all_true) > 0:
    # Общая точность
    overall_accuracy = accuracy_score(all_true, all_pred)
    print(f"Общая точность (Accuracy): {overall_accuracy:.4f}")
    
    # Коэффициент Каппа Коэна
    kappa = cohen_kappa_score(all_true, all_pred)
    print(f"Коэффициент Каппа Коэна: {kappa:.4f}")
    
    # Macro метрики
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_true, all_pred, average='macro', zero_division=0
    )
    print(f"Macro F1: {f1_macro:.4f}")
    print(f"Macro Precision: {precision_macro:.4f}")
    print(f"Macro Recall: {recall_macro:.4f}")
    
    # Weighted метрики
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        all_true, all_pred, average='weighted', zero_division=0
    )
    print(f"Weighted F1: {f1_weighted:.4f}")
else:
    print("Нет валидных пар для расчета метрик!")
    overall_accuracy = kappa = f1_macro = precision_macro = recall_macro = f1_weighted = 0.0

# ============================================
# ШАГ 7: Метрики по каждой черте отдельно
# ============================================
print("\n" + "="*80)
print("МЕТРИКИ ПО КАЖДОЙ ЧЕРТЕ ЛИЧНОСТИ")
print("="*80)

trait_metrics = {}

for trait in TRAITS:
    # Фильтруем пары, где оба значения не None
    valid_pairs = [(t, p) for t, p in zip(true_labels_dict[trait], pred_labels_dict[trait])
                   if t is not None and p is not None]
    
    if not valid_pairs:
        print(f"\n{trait}: недостаточно данных")
        continue
    
    true_filtered = [t for t, p in valid_pairs]
    pred_filtered = [p for t, p in valid_pairs]
    
    acc = accuracy_score(true_filtered, pred_filtered)
    
    print(f"\n--- {trait} ---")
    print(f"Точность (Accuracy): {acc:.4f}")
    
    # Проверяем наличие всех классов
    unique_classes = set(true_filtered + pred_filtered)
    print(f"Присутствующие классы: {sorted(unique_classes)}")
    
    # Classification Report
    print("\nClassification Report:")
    try:
        # Определяем метки классов на основе присутствующих
        class_labels = sorted(unique_classes)
        target_names = [f'класс {i}' for i in class_labels]
        
        report = classification_report(
            true_filtered, 
            pred_filtered, 
            labels=class_labels,
            target_names=target_names,
            zero_division=0
        )
        print(report)
    except Exception as e:
        print(f"Ошибка при создании отчета: {e}")
    
    # Матрица ошибок
    try:
        cm = confusion_matrix(true_filtered, pred_filtered, labels=class_labels)
        cm_df = pd.DataFrame(
            cm, 
            index=[f'True: {i}' for i in class_labels],
            columns=[f'Pred: {i}' for i in class_labels]
        )
        print("Матрица ошибок:")
        print(cm_df)
    except Exception as e:
        print(f"Ошибка при создании матрицы ошибок: {e}")
    
    trait_metrics[trait] = {
        'accuracy': acc,
        'confusion_matrix': cm.tolist() if 'cm' in locals() else []
    }

# ============================================
# ШАГ 8: Сводная таблица результатов
# ============================================
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА")
print("="*80)

summary_data = []
for trait in TRAITS:
    valid_pairs = [(t, p) for t, p in zip(true_labels_dict[trait], pred_labels_dict[trait])
                   if t is not None and p is not None]
    
    if valid_pairs:
        true_filtered = [t for t, p in valid_pairs]
        pred_filtered = [p for t, p in valid_pairs]
        
        acc = accuracy_score(true_filtered, pred_filtered)
        prec, rec, f1, _ = precision_recall_fscore_support(
            true_filtered, pred_filtered, average='macro', zero_division=0
        )
    else:
        acc = prec = rec = f1 = 0.0
    
    summary_data.append({
        'Черта': trait,
        'Accuracy': f"{acc:.4f}",
        'Precision (macro)': f"{prec:.4f}",
        'Recall (macro)': f"{rec:.4f}",
        'F1 (macro)': f"{f1:.4f}",
        'Valid pairs': len(valid_pairs) if valid_pairs else 0
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# ============================================
# ШАГ 9: Сохранение результатов
# ============================================
output_results = {
    'overall_metrics': {
        'accuracy': float(overall_accuracy),
        'cohen_kappa': float(kappa),
        'macro_f1': float(f1_macro),
        'weighted_f1': float(f1_weighted),
        'macro_precision': float(precision_macro),
        'macro_recall': float(recall_macro),
        'total_valid_pairs': len(all_true)
    },
    'trait_metrics': trait_metrics,
    'predictions': []
}

# Добавляем предсказания
for i, result in enumerate(results):
    pred_clean = {}
    for trait in TRAITS:
        pred_clean[trait] = result['predicted_labels'].get(trait)
    
    output_results['predictions'].append({
        'user_id': result['user_id'],
        'true_labels': {t: normalize_label(result['true_labels'].get(t)) for t in TRAITS},
        'predicted_labels': {t: normalize_label(result['predicted_labels'].get(t)) for t in TRAITS},
        'correct': {
            trait: normalize_label(result['true_labels'].get(trait)) == normalize_label(result['predicted_labels'].get(trait))
            for trait in TRAITS
        }
    })

output_path = f"{LORA_WEIGHTS_PATH}/../test_metrics_fixed.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output_results, f, ensure_ascii=False, indent=2)

print(f"\nРезультаты сохранены в {output_path}")

# ============================================
# ШАГ 10: Дополнительная диагностика
# ============================================
print("\n" + "="*80)
print("ДИАГНОСТИКА")
print("="*80)

print("\nРаспределение истинных меток:")
for trait in TRAITS:
    true_vals = [v for v in true_labels_dict[trait] if v is not None]
    if true_vals:
        unique, counts = np.unique(true_vals, return_counts=True)
        print(f"  {trait}: {dict(zip(unique, counts))}")

print("\nРаспределение предсказанных меток:")
for trait in TRAITS:
    pred_vals = [v for v in pred_labels_dict[trait] if v is not None]
    if pred_vals:
        unique, counts = np.unique(pred_vals, return_counts=True)
        print(f"  {trait}: {dict(zip(unique, counts))}")

print("\nПримеры успешных и неуспешных предсказаний:")
for i in range(min(5, len(results))):
    print(f"\n--- Пример {i+1} ---")
    print(f"User ID: {results[i]['user_id']}")
    print("Истинные vs Предсказанные:")
    for trait in TRAITS:
        true_val = normalize_label(results[i]['true_labels'].get(trait))
        pred_val = normalize_label(results[i]['predicted_labels'].get(trait))
        match = "✓" if true_val == pred_val else "✗"
        print(f"  {trait}: True={true_val}, Pred={pred_val} {match}")

Загружено тестовых примеров: 133

Пример структуры данных:
{
  "user_id": 519593,
  "prompt": "# Задача\nВыполните анализ личности пользователя социальной сети на основе его публикаций. Оцените каждую из пяти черт личности по шкале: 0 - низкий уровень, 1 - средний уровень, 2 - высокий уровень. Ответ предоставьте строго в JSON-формате без дополнительного текста.\n\n# Описание черт личности\n- Открытость опыту: любознательность, креативность, воображение, интерес к новым идеям и эстетическим переживаниям\n- Добросовестность: организованность, дисциплина, надёжность, целеустремлённость, внимание к деталям\n- Экстраверсия: общительность, энергичность, напористость, склонность к социальным взаимодействиям\n- Доброжелательность: эмпатия, готовность помогать, доверие к людям, склонность к сотрудничеству\n- Нейротизм: эмоциональная нестабильность, тревожность, раздражительность, подверженность стрессу\n# Информация о пользователе\nПол: Female\nВозраст: 39.0\n\n# Публикации пользователя\nМоим д

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Загрузка LoRA весов...
Модель загружена успешно!

Генерация предсказаний для тестового набора...


100%|████████████████████████████████████████████████████████████████████████████████| 133/133 [14:45<00:00,  6.66s/it]

Успешных предсказаний: 133/133
Неудачных предсказаний: 0

Анализ формата истинных меток:
Тип значений: <class 'int'>
Пример значений: [0, 0, 1]

Проверка конвертированных данных:
открытость_опыту:
  Валидных истинных: 133
  Валидных предсказанных: 133
  Уникальные истинные: {0, 1, 2}
  Уникальные предсказанные: {0, 2}

МЕТРИКИ ОЦЕНКИ КЛАССИФИКАЦИИ
Всего валидных пар: 665
Общая точность (Accuracy): 0.3113
Коэффициент Каппа Коэна: -0.0340
Macro F1: 0.2958
Macro Precision: 0.2996
Macro Recall: 0.3112
Weighted F1: 0.2968

МЕТРИКИ ПО КАЖДОЙ ЧЕРТЕ ЛИЧНОСТИ

--- открытость_опыту ---
Точность (Accuracy): 0.3459
Присутствующие классы: [0, 1, 2]

Classification Report:
              precision    recall  f1-score   support

     класс 0       0.36      0.93      0.51        46
     класс 1       0.00      0.00      0.00        36
     класс 2       0.25      0.06      0.10        51

    accuracy                           0.35       133
   macro avg       0.20      0.33      0.20       133
weight